# Convert JSON to Parquet with chDB

In-process ClickHouse engine. Run `./generate.sh` first to create `./data/events.json`.
Companion article: https://clickhouse.com/resources/engineering/convert-json-to-parquet

In [ ]:
import os
import chdb

os.chdir("data")

Convert JSON to Parquet in one statement — the same `SELECT ... INTO OUTFILE ... FORMAT Parquet` you'd run on the CLI.

In [ ]:
chdb.query(
    "SELECT * FROM file('events.json') "
    "INTO OUTFILE 'events_chdb.parquet' TRUNCATE FORMAT Parquet"
)
print("wrote events_chdb.parquet")

Schema carried from JSON into Parquet — the nested `geo` object stays a typed `Tuple`.

In [ ]:
chdb.query("DESCRIBE file('events_chdb.parquet')", "DataFrame")[["name", "type"]]

Read the typed columns back, including the nested fields.

In [ ]:
print(chdb.query(
    "SELECT event_id, geo.country AS country, geo.city AS city, tags, amount "
    "FROM file('events_chdb.parquet') ORDER BY event_id LIMIT 5",
    "PrettyCompact",
))

Or aggregate on the nested column and pull the result straight into a pandas DataFrame.

In [ ]:
chdb.query(
    "SELECT geo.country AS country, count() AS events, round(sum(amount), 2) AS total "
    "FROM file('events_chdb.parquet') GROUP BY country ORDER BY total DESC",
    "DataFrame",
)